In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="4"

In [2]:
import sys

# sys.path.append("/mnt/data-poseidon/sweta/chat-translation-generation/wmt24-chat-translation")
sys.path.append("/mnt/data-poseidon/sweta/context-aware-mt/chat-translation-generation/wmt24-chat-translation/scripts")

In [ ]:
from run_context_comet_mbr import *

In [4]:
def get_candidates(df, lp, split="dev", n=100, model_name="TowerInstruct-7B-w-chat-empty-sys", context_type="full_context_empty_sys", data_name="wmt24_chat"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')
  if split=="train":
     xx_yy_generations = read_file(f"../training_set/TowerChat/{source_lang}-{target_lang}/100_epsilon_candidates.txt")
     yy_xx_generations = read_file(f"../training_set/TowerChat/{target_lang}-{source_lang}/100_epsilon_candidates.txt")
  else:
    try:
       xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
       yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")
    except:
       xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
       yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")


  xx_yy_generations = np.array(xx_yy_generations).reshape((len(xx_yy_generations)//n, n))
  yy_xx_generations = np.array(yy_xx_generations).reshape((len(yy_xx_generations)//n, n))

  assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
  translations = []
  j,k=0,0

  for i, row in tqdm(df.iterrows()):
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(xx_yy_generations[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(yy_xx_generations[k])
      k+=1
  return np.array(translations)


def get_dataframe(split, lang_pair, candidate_file, data_name='wmt24_chat', model_name='TowerInstruct-7B-w-chat-empty-sys', context_type="full_context_empty_sys"):
    df = pd.read_csv(f"../tacl_results/{data_name}/{split}.{lang_pair}.csv",
                    keep_default_na=False,
        na_values=["NaN"],)
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    if candidate_file:
        outputs = read_file(candidate_file)
        output_reshaped = np.array(outputs).reshape((len(outputs)//len(df),  len(df))).T
    else:
        output_reshaped = get_candidates(df, lang_pair, split, data_name=data_name, model_name=model_name, context_type=context_type)

    if split=="dev" or split=="test":
        df["greedy"] = get_translations(df, lang_pair, 
                                    context_type=context_type, 
                                    model_name=model_name, 
                                    data_name=data_name,
                                    split=split)
    return df, output_reshaped


In [5]:
# use_context = True
# context_type = "hyp-only"
# context_size = 2
# batch_size = 128

# df, output_reshaped = get_dataframe("dev", "en-de")
# n_sent, num_samples = output_reshaped.shape


# sources, outputs = get_source_and_translations(df, 
#                                                 output_reshaped, 
#                                                 use_context, 
#                                                 context_type, 
#                                                 context_size)

# check oracle

In [6]:
# # check if greedy is read correctly
# df["comet-greedy"] = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"].to_list(), df["greedy"], df["reference"].to_list() )], 
#         batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
# for lp, lp_df in df.groupby("lp"):
#     print(lp, lp_df["comet-greedy"].mean())

In [7]:
def build_embeddings(sources, translations, references, comet_model, batch_size):

    src_batches = [
        sources[i : i + batch_size] for i in range(0, len(sources), batch_size)
    ]
    src_inputs = [comet_model.encoder.prepare_sample(batch) for batch in src_batches]
    mt_batches = [
        translations[i : i + batch_size]
        for i in range(0, len(translations), batch_size)
    ]
    mt_inputs = [comet_model.encoder.prepare_sample(batch) for batch in mt_batches]

    ref_batches = [
        translations[i : i + batch_size]
        for i in range(0, len(references), batch_size)
    ]
    ref_inputs = [comet_model.encoder.prepare_sample(batch) for batch in ref_batches]

    src_embeddings = []
    with torch.no_grad():
        for batch in src_inputs:
            input_ids = batch["input_ids"].to(comet_model.device)
            attention_mask = batch["attention_mask"].to(comet_model.device)
            src_embeddings.append(
                comet_model.get_sentence_embedding(input_ids, attention_mask)
            )
    src_embeddings = torch.vstack(src_embeddings)

    mt_embeddings = []
    with torch.no_grad():
        for batch in tqdm(mt_inputs, desc="Encoding sentences...", dynamic_ncols=True):
            input_ids = batch["input_ids"].to(comet_model.device)
            attention_mask = batch["attention_mask"].to(comet_model.device)
            mt_embeddings.append(
                comet_model.get_sentence_embedding(input_ids, attention_mask)
            )
    mt_embeddings = torch.vstack(mt_embeddings)

    ref_embeddings = []
    with torch.no_grad():
        for batch in tqdm(ref_inputs, desc="Encoding sentences...", dynamic_ncols=True):
            input_ids = batch["input_ids"].to(comet_model.device)
            attention_mask = batch["attention_mask"].to(comet_model.device)
            ref_embeddings.append(
                comet_model.get_sentence_embedding(input_ids, attention_mask)
            )
    ref_embeddings = torch.vstack(ref_embeddings)

    return src_embeddings, mt_embeddings, ref_embeddings

In [ ]:
# check if oracle scores suggest benefit of reranking
import numpy as np
num_repeats=100
df, output_reshaped = get_dataframe("train", "en-de", None)
n_sent, num_samples = output_reshaped.shape
outputs = list(chain.from_iterable(output_reshaped))
comet_metric.eval()
comet_metric.to(0)
src_embeddings, mt_embeddings, ref_embeddings = build_embeddings(df["source"].to_list(), outputs, df["reference"].to_list(), comet_metric, 256)


In [ ]:
n_sent = len(df["source"].to_list())
num_samples = num_repeats
mbr_matrix = torch.zeros(n_sent, num_samples)
mt_embeddings = mt_embeddings.reshape(n_sent, num_samples, -1)

with torch.no_grad():
    # Loop over all source sentences
    for i in tqdm(
        range(mbr_matrix.shape[0]), desc="all Scores...", dynamic_ncols=True
    ):
        mbr_matrix[i] = comet_metric.estimate(src_embeddings[i, :].repeat(num_repeats, 1), mt_embeddings[i], ref_embeddings[i, :].repeat(num_repeats, 1))["score"]

col_argmax = np.argmax(mbr_matrix, axis=1)
df["comet-oracle"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]
df["comet-oracle-score"] = np.max(mbr_matrix, axis=1)

In [ ]:
for lp, lp_df in df.groupby("lp"):
    print(lp, lp_df["comet-oracle-score"].mean())

# Run MBR

In [ ]:
mbr_matrix = run_mbr(comet_metric, 
                        sources, 
                        outputs, 
                        n_sent,
                        num_samples,
                        context_type=context_type,
                        batch_size=batch_size, 
                        use_context=use_context)

In [ ]:
mbr_matrix.shape

In [11]:
import pickle

with open("/mnt/data-poseidon/sweta/chat-translation-generation/wmt24-chat-translation/mbr_matrix.pkl", "rb") as f:
    mbr_matrix_2 = pickle.load(f)

In [8]:
# df["output-select"]  = read_file("/mnt/data-poseidon/sweta/chat-translation-generation/wmt24-chat-translation/mbr_outputs_dev/full_context_empty_sys/wmt24_chat_dev.en-de/comet_eps_context_source_across.out.txt")

In [ ]:
import sacrebleu
comet_metric.use_context = False
col_argmax = np.argmax(mbr_matrix, axis=1)
df["output-select"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]

for col in ["output-select", "greedy"]:
    df[f"{col}-comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df[col], df["reference"])], 
        batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
    df[f"{col}-chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df[col].to_list(), df["reference"].to_list())]

    for lp, lp_df in df.groupby("lp"):
        print(col, lp, lp_df[f"{col}-comet"].mean(), lp_df[f"{col}-chrf"].mean())


# Run Kiwi

In [21]:
sources = list(chain.from_iterable(np.array(df["source"].to_list()* output_reshaped.shape[1]).reshape(output_reshaped.T.shape).T))


In [19]:
from comet import download_model, load_from_checkpoint
checkpoint_path = download_model("Unbabel/wmt22-comet-da")
comet_metric = load_from_checkpoint(checkpoint_path)


In [ ]:
# checkpoint_path = download_model("Unbabel/wmt23-cometkiwi-da-xl")
# comet_metric = load_from_checkpoint(checkpoint_path)

In [ ]:
qe_scores = comet_metric.predict([{"mt": y, "src": x} for x, y in zip(sources, outputs)], 
            batch_size=128, gpus=1, progress_bar=True, devices=[0])['scores']
qe_scores = np.array(qe_scores).reshape(output_reshaped.shape)

In [26]:
col_argmax = np.argmax(qe_scores, axis=1)
df["output-select"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]

In [27]:
import sacrebleu

In [ ]:
checkpoint_path = download_model("Unbabel/wmt22-comet-da")
comet_metric = load_from_checkpoint(checkpoint_path)

for col in ["output-select", "greedy"]:
    df[f"{col}-comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df[col], df["reference"])], 
        batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
    df[f"{col}-chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df[col].to_list(), df["reference"].to_list())]

    for lp, lp_df in df.groupby("lp"):
        print(col, lp, lp_df[f"{col}-comet"].mean(), lp_df[f"{col}-chrf"].mean())


# Check Overlap candidates with and without context

In [1]:
import pandas as pd
import numpy as np

def read_file(fname, unescape_newline=True):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    if unescape_newline:
      output = [l.replace("\\n", "\n") for l in output]
    return output

def get_candidates(df, lp, split="dev", n=100, key="full_context_empty_sys"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')

  if key=="full_context_empty_sys":
     split = split.replace('test', 'test_blind')
  
  xx_yy_generations = read_file(f"../candidates/{key}/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_{split}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
  yy_xx_generations = read_file(f"../candidates/{key}/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_{split}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")

  xx_yy_generations = np.array(xx_yy_generations).reshape((len(xx_yy_generations)//n, n))
  yy_xx_generations = np.array(yy_xx_generations).reshape((len(yy_xx_generations)//n, n))

  assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
  translations = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(list(xx_yy_generations[j]))
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(list(yy_xx_generations[k]))
      k+=1
  return translations


In [ ]:
split = "test"

for lang in ["de", "fr", "pt", "ko", "nl"]:
        lang_pair = f"en-{lang}"

        df = pd.read_csv(f"../paper_results/{split}.{lang_pair}.csv")

        full_context_output_reshaped = get_candidates(df, lang_pair, split)
        no_context_output_reshaped = get_candidates(df, lang_pair, split, key="no_context_empty_sys")

        print("Greedy", len(df[df["greedy-7b-ft-w-context"] == df["greedy-7b-ft-wo-context"]])/len(df))

        count_overlap = []
        for i in range(len(full_context_output_reshaped)):
                count_overlap.append(len(set(full_context_output_reshaped[i]).intersection(set(no_context_output_reshaped[i])))/100)

        print("Average cands overlap",  sum(count_overlap) / len(full_context_output_reshaped))

# Check training data Quality

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [2]:
from comet import download_model, load_from_checkpoint
checkpoint_path = download_model("Unbabel/wmt22-comet-da")
comet_metric = load_from_checkpoint(checkpoint_path)


/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 13340.66it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../scratch-artemis/sweta/cache/models--Unbabel--wmt22-comet-da/snapshots/f49d328952c3470eff6bb6f545d62bfdb6e66304/checkpoints/model.ckpt`
/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoder

In [3]:
comet_metric.eval()

RegressionMetric(
  (encoder): XLMREncoder(
    (model): XLMRobertaModel(
      (embeddings): XLMRobertaEmbeddings(
        (word_embeddings): Embedding(250002, 1024, padding_idx=1)
        (position_embeddings): Embedding(514, 1024, padding_idx=1)
        (token_type_embeddings): Embedding(1, 1024)
        (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): XLMRobertaEncoder(
        (layer): ModuleList(
          (0-23): 24 x XLMRobertaLayer(
            (attention): XLMRobertaAttention(
              (self): XLMRobertaSelfAttention(
                (query): Linear(in_features=1024, out_features=1024, bias=True)
                (key): Linear(in_features=1024, out_features=1024, bias=True)
                (value): Linear(in_features=1024, out_features=1024, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): XLMRobertaSelfOutput(
           

In [4]:
import pandas as pd
from IPython.display import display
import sacrebleu

for lang in ["de", "fr", "pt", "ko", "nl"]:
        lang_pair = f"en-{lang}"

        df = pd.read_csv(f"../mbr_outputs/mbr_outputs_train/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_train.en-{lang}/comet_eps_context_source_w2.csv")
        print(len(df))
        
        df = df.dropna()
        # dropped_rows = df[df.isnull().any(axis=1)]
        # print(dropped_rows)
        print(len(df))

        df[f"chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df["output-select"].to_list(), df["reference"].to_list())]
        df[f"comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df["output-select"], df["reference"])], 
                        batch_size=128, gpus=1, progress_bar=True, devices=[0])['scores']
        print(lang, df[f"chrf"].mean(), df["comet"].mean())

        df = pd.read_csv(f"../mbr_outputs/mbr_outputs_train/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_train.en-{lang}/comet_eps_context_source_w0.csv")
        
        print(len(df))
        df = df.dropna()
        print(len(df))

        df[f"chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df["output-select"].to_list(), df["reference"].to_list())]
        df[f"comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df["output-select"], df["reference"])], 
                        batch_size=128, gpus=1, progress_bar=True, devices=[0])['scores']
        print(lang, df[f"chrf"].mean(), df["comet"].mean())


17805
17805


/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 140/140 [01:37<00:00,  1.44it/s]
Traceback (most recent call last):
Traceback (most recent ca

de 84.87544103006853 0.941115376929894
17805
17805


/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 140/140 [01:34<00:00,  1.49it/s]
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers

de 84.67725492922361 0.9422511930999311
15026
15024


...

ko 81.00181803067055 0.9550714192609945
15463
15462


/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 121/121 [01:06<00:00,  1.81it/s]
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers

nl 85.83244751296937 0.9478266891552242
15463
15462


/mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/qe-bias-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 121/121 [01:09<00:00,  1.73it/s]
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/opt/spack-software/installs/gcc-11.4.0/python-3.10.14-hxqv6dhp5swvqmkybwfovmbveyonvniq/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers

nl 85.68176392959148 0.9481572565190034
